In [ ]:
import openai
import os
import sys
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm, trange

# Set your OpenAI API key



sys.path.append("../../")
import biked_commons
from biked_commons.resource_utils import split_datasets_path, models_and_scalers_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.design_evaluation import construct_tensor_evaluator, get_standard_evaluations
from biked_commons.transformation import one_hot_encoding

In [2]:
filedir = "openai_files"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [3]:
data = pd.read_csv(split_datasets_path("bike_bench_mixed_modality.csv"), index_col=0)
data_oh = one_hot_encoding.encode_to_continuous(data)
data_tens = torch.tensor(data_oh.values, dtype=torch.float32).to(device)

In [4]:
def get_condition_by_idx(idx=0):
    rider_condition = conditioning.sample_riders(10, split="test")
    use_case_condition = conditioning.sample_use_case(10, split="test")
    text_embeddings = conditioning.sample_text(10, split="test")
    condition = {"Rider": rider_condition[idx], "Use Case": use_case_condition[idx], "Text": text_embeddings[idx]}
    return condition

def get_conditions_10k():
    rider_condition = conditioning.sample_riders(10000, split="test")
    use_case_condition = conditioning.sample_use_case(10000, split="test")
    text_embeddings = conditioning.sample_text(10000, split="test")
    conditions = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_embeddings}
    return conditions


def build_text_condition(condition):

    rc = condition["Rider"]
    rc_text = f"Rider Body Dimensions: Upper leg length - {rc[0]}, Lower leg length - {rc[1]}, Arm length - {rc[2]}, Torso length - {rc[3]}, Neck and head length - {rc[4]}, Torso width - {rc[5]}"
    tc = condition["Text"]
    uc = condition["Use Case"]
    #get argmax of uc
    uc = uc.argmax()
    if uc == 0:
        uc_text = "Use Case: Road Biking"
    elif uc == 1:
        uc_text = "Use Case: Mountain Biking"
    elif uc == 2:
        uc_text = "Use Case: Commuting"
    tc_text = "Marketing Description: " + tc
    full_text = f"{rc_text}. {uc_text}. {tc_text}"
    return full_text

cond = get_condition_by_idx(0)
cond_text = build_text_condition(cond)

In [5]:
ds_len = len(data)

text_conditions = []
rider_condition = conditioning.sample_riders(ds_len, split="train", randomize=True)
use_case_condition = conditioning.sample_use_case(ds_len, split="train", randomize=True)
text_embeddings = conditioning.sample_text(ds_len, split="train", randomize=True)
for i in trange(ds_len):
    conditions = {"Rider": rider_condition[i], "Use Case": use_case_condition[i], "Text": text_embeddings[i]}
    text_conditions.append(build_text_condition(conditions))
print("Text conditions built")

with open(os.path.join(filedir, "text_conditions.txt"), "w") as f:
    for item in text_conditions:
        f.write("%s\n" % item)


100%|██████████| 4500/4500 [00:00<00:00, 31774.76it/s]

Text conditions built


In [13]:
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_embeddings}
eval_fns = get_standard_evaluations(device, aesthetics_mode="Text")
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(eval_fns, data_oh.columns)
eval_scores = evaluator(data_tens, condition)
eval_scores_df = pd.DataFrame(eval_scores.cpu().detach().numpy(), columns=requirement_names)
eval_scores_df.to_csv(os.path.join(filedir, "eval_scores.csv"), index=False)

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
SYSTEM_PROMPT = """
# I will ask you to create some bicycle designs. The bicycle designs are subject to a text prompt, some rider dimensions, and a use case. 
# Each design is defined by 71 variables, which I will describe. Some of these are categorical variables, and I will provide you with the possible values for these variables. Others are continuous.
# The design variables are as follows: 



# """.strip()

In [ ]:
# # File paths
# csv1_path = "openai_files/condition.csv"
# csv2_path = "openai_files/data.csv"
# csv3_path = "openai_files/result.csv"
# output_path = "output.csv"

# # Load CSV contents as strings
# with open(csv1_path, "r") as f:
#     csv1_data = f.read()

# with open(csv2_path, "r") as f:
#     csv2_data = f.read()

# # System prompt (model behavior setup)
# SYSTEM_PROMPT = """
# I will ask you to create some bicycle designs. The bicycle designs are subject to a text prompt, some rider dimensions, and a use case. 
# The design consists of 
# """.strip()

# # User instructions (custom logic)
# USER_INSTRUCTIONS = """
# You are provided with two CSV files.

# Please:
# - Join them on the column 'id'
# - Keep only rows where 'status' == 'active'
# - Include only the columns 'id', 'name', and 'score'
# - Return the result as plain CSV with no extra explanation, no Markdown, and no code block formatting
# """.strip()

# # === END CONFIG SECTION ===

# # Prepare the prompt messages
# messages = [
#     {"role": "system", "content": SYSTEM_PROMPT},
#     {
#         "role": "user",
#         "content": (
#             f"CSV File 1:\n{csv1_data}\n\n"
#             f"CSV File 2:\n{csv2_data}\n\n"
#             f"{USER_INSTRUCTIONS}"
#         )
#     }
# ]

# # Call the model
# response = openai.ChatCompletion.create(
#     model="gpt-4",
#     messages=messages,
#     temperature=0,
# )

# # Save the output


In [ ]:
csv_output = response['choices'][0]['message']['content'].strip()

with open(output_path, "w") as f:
    f.write(csv_output)

print(f"✅ Saved generated CSV to: {output_path}")